# MLPerf-Style Mini Language Model Benchmark for CPU, GPU, and TPU on GCP

This notebook is an educational mini benchmark. It is not an official MLPerf submission, does not use the official MLPerf rules, and should not be used to rank vendors or hardware platforms.

It is the language-model sibling of the vision mini benchmark notebook (`mlperf_style_vision_benchmark.ipynb`). Instead of a small CNN on images, it trains and runs inference on a small GPT-style decoder-only transformer, with a `DATASET_NAME` switch across three text datasets of increasing size: WikiText-2, WikiText-103, and TinyStories.

The notebook benchmarks only the runtime where it is currently running. To compare CPU, GPU, and TPU (and across datasets), run this notebook once per `DATASET_NAME` per runtime and combine the CSV result files.

## Cost Warning and Recommended Run Order

This notebook does not launch GCP resources. It only uses the CPU, GPU, or TPU attached to the current Jupyter runtime.

Language models are much more compute- and memory-hungry than the small CNN used in the vision notebook. Use this recommended order:

1. Run the notebook on CPU first, with `DATASET_NAME = "wikitext2"` and `BENCHMARK_PROFILE = "quick"`. WikiText-2 (~2M tokens) is small enough to be a fast pipeline sanity check -- confirm dataset loading, tokenization, packing, and the four training/inference batch-size combinations all run before spending accelerator budget.
2. Run a small TPU v5e experiment next, using `BENCHMARK_PROFILE = "expanded"` or `"accelerator_stress"`.
3. Run H100 only after the CPU and TPU pipeline works.
4. `BENCHMARK_PROFILE = "accelerator_stress"` uses a GPT-2-small-sized model (~124M parameters, 12 layers/12 heads/768 dim) with `sequence_length = 512`. This is intentionally sized so that roughly one epoch of WikiText-103 (~1e8 tokens) fits within a single-chip TPU v5e run in about an hour, per the notebook author's budget target -- adjust `MAX_TRAIN_TOKENS` if your budget or quota differs.
5. TinyStories (~4.7e8 tokens) is larger than WikiText-103; use `MAX_TRAIN_TOKENS` to deliberately use only a slice of it rather than the full dataset, unless you intend a longer run.
6. Stop or delete expensive GPU and TPU resources immediately after the experiment.
7. Save CSV results so you do not need to repeat expensive runs.

Warning: This notebook is intentionally small. Mini benchmark results should not be overgeneralized to full-scale MLPerf or production LLM pretraining results.

In [ ]:
import csv
import datetime as dt
import gc
import json
import math
import os
import pathlib
import platform
import statistics
import subprocess
import sys
import time
import traceback
import urllib.request
import uuid
import zipfile
from typing import Any, Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

try:
    import jax
    import jax.numpy as jnp
except ImportError as exc:
    raise ImportError(
        "This notebook requires JAX. Install the JAX build that matches your CPU, GPU, or TPU runtime."
    ) from exc

# Use a benchmark profile to control model size, cost, and runtime.
# quick: tiny 2-layer model, meant for WikiText-2 pipeline sanity checks on CPU or a new runtime.
# expanded: a mid-size model (6 layers, 512 dim) for a more useful accelerator comparison.
# accelerator_stress: a GPT-2-small-sized model (~124M params, 12 layers/12 heads/768 dim,
#                      sequence_length=512) for H100/TPU v5e after the pipeline is validated.
BENCHMARK_PROFILE = "accelerator_stress"

BENCHMARK_PROFILES = {
    "quick": {
        "batch_sizes": [1, 4, 8, 16, 32],
        "train_warmup_steps": 3,
        "train_measure_steps": 10,
        "inference_warmup_steps": 3,
        "inference_measure_steps": 20,
        "max_train_tokens": 200_000,
        "max_test_tokens": 50_000,
        "sequence_length": 128,
        "n_layer": 2,
        "n_head": 2,
        "n_embd": 64,
    },
    "expanded": {
        "batch_sizes": [1, 8, 16, 32, 64, 128],
        "train_warmup_steps": 5,
        "train_measure_steps": 30,
        "inference_warmup_steps": 5,
        "inference_measure_steps": 75,
        "max_train_tokens": 20_000_000,
        "max_test_tokens": 2_000_000,
        "sequence_length": 512,
        "n_layer": 6,
        "n_head": 8,
        "n_embd": 512,
    },
    "accelerator_stress": {
        "batch_sizes": [1, 8, 16, 32, 64, 128, 256],
        "train_warmup_steps": 10,
        "train_measure_steps": 100,
        "inference_warmup_steps": 10,
        "inference_measure_steps": 250,
        "max_train_tokens": 100_000_000,
        "max_test_tokens": 5_000_000,
        "sequence_length": 512,
        "n_layer": 12,
        "n_head": 12,
        "n_embd": 768,
    },
}

if BENCHMARK_PROFILE not in BENCHMARK_PROFILES:
    raise ValueError(f"Unknown BENCHMARK_PROFILE={BENCHMARK_PROFILE!r}")

PROFILE_CONFIG = BENCHMARK_PROFILES[BENCHMARK_PROFILE]

# Keep the run controlled for GCP experiments. Increase the profile only after a quick run succeeds.
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RANDOM_SEED = 7

# Pick which text dataset to benchmark. Run the notebook once per dataset per runtime, then
# combine lm_benchmark_results_all.csv rows (already labeled by the "dataset" column) to compare.
# wikitext2:   ~2M tokens. Fast pipeline sanity check; use with BENCHMARK_PROFILE="quick".
# wikitext103: ~1e8 tokens. Standard LM benchmark scale; fits accelerator_stress's default budget.
# tinystories: ~4.7e8 tokens. Simple vocabulary; small models show clearly decreasing loss quickly.
#              MAX_TRAIN_TOKENS below deliberately uses only a slice unless you raise it.
DATASET_NAME = "wikitext103"

if DATASET_NAME not in ("wikitext2", "wikitext103", "tinystories"):
    raise ValueError(f"Unknown DATASET_NAME={DATASET_NAME!r}. Choose wikitext2, wikitext103, or tinystories.")

MODEL_NAME = "SmallJAXGPT"
FRAMEWORK_NAME = "JAX/XLA"

# Use float32 for the default apples-to-apples educational baseline.
# For a follow-up accelerator-focused experiment, students can test bfloat16 carefully and label it clearly.
COMPUTE_DTYPE = jnp.float32

# This notebook uses a byte-level tokenizer (each of the 256 possible byte values is one token)
# instead of a real subword/BPE tokenizer. This keeps the notebook dependency-free and dataset-
# agnostic, at the cost of realism: real LLM pretraining uses BPE vocabularies of 32k-100k+ tokens,
# which changes effective context length and throughput-per-token comparisons. See the Dataset
# Choice section below and the Interpretation Guide at the end for details.
VOCAB_SIZE = 256

SEQUENCE_LENGTH = PROFILE_CONFIG["sequence_length"]
N_LAYER = PROFILE_CONFIG["n_layer"]
N_HEAD = PROFILE_CONFIG["n_head"]
N_EMBD = PROFILE_CONFIG["n_embd"]
if N_EMBD % N_HEAD != 0:
    raise ValueError(f"n_embd={N_EMBD} must be divisible by n_head={N_HEAD}.")
HEAD_DIM = N_EMBD // N_HEAD

BATCH_SIZES = PROFILE_CONFIG["batch_sizes"]
TRAIN_WARMUP_STEPS = PROFILE_CONFIG["train_warmup_steps"]
TRAIN_MEASURE_STEPS = PROFILE_CONFIG["train_measure_steps"]
INFERENCE_WARMUP_STEPS = PROFILE_CONFIG["inference_warmup_steps"]
INFERENCE_MEASURE_STEPS = PROFILE_CONFIG["inference_measure_steps"]
LEARNING_RATE = 0.0006

MAX_TRAIN_TOKENS = PROFILE_CONFIG["max_train_tokens"]
MAX_TEST_TOKENS = PROFILE_CONFIG["max_test_tokens"]
ALLOW_DATASET_DOWNLOAD = True
ALLOW_SYNTHETIC_FALLBACK = True
DATA_ROOT = pathlib.Path("data/lm") / DATASET_NAME

RUN_TRAINING_BENCHMARK = True
RUN_INFERENCE_BENCHMARK = True

TRAINING_RESULTS_PATH = pathlib.Path("lm_benchmark_results_training.csv")
INFERENCE_RESULTS_PATH = pathlib.Path("lm_benchmark_results_inference.csv")
ALL_RESULTS_PATH = pathlib.Path("lm_benchmark_results_all.csv")

RESULT_COLUMNS = [
    "timestamp",
    "run_id",
    "device_type",
    "hardware_model",
    "accelerator_count",
    "framework",
    "framework_version",
    "task_type",
    "dataset",
    "model",
    "batch_size",
    "sequence_length",
    "vocab_size",
    "compile_time_seconds",
    "warm_up_time_seconds",
    "average_latency_ms",
    "median_latency_ms",
    "throughput_samples_per_second",
    "tokens_per_second",
    "total_wall_time_seconds",
    "number_of_iterations",
    "first_run_overhead_seconds",
    "memory_used_mb",
    "numeric_precision",
    "hardware_generation_note",
    "status",
    "notes",
]

print(f"Run ID: {RUN_ID}")
print(f"Python: {sys.version.split()[0]}")
print(f"JAX: {jax.__version__}")
print(f"Default JAX backend: {jax.default_backend()}")
print(f"Benchmark profile: {BENCHMARK_PROFILE}")
print(f"Dataset: {DATASET_NAME}")
print(f"Model: n_layer={N_LAYER}, n_head={N_HEAD}, n_embd={N_EMBD}, sequence_length={SEQUENCE_LENGTH}, vocab_size={VOCAB_SIZE}")
print(f"Batch sizes (sequences per batch): {BATCH_SIZES}")
print(f"Training warm-up/measured steps: {TRAIN_WARMUP_STEPS}/{TRAIN_MEASURE_STEPS}")
print(f"Inference warm-up/measured steps: {INFERENCE_WARMUP_STEPS}/{INFERENCE_MEASURE_STEPS}")


## Environment Detection

This section records the exact runtime context. Hardware labels matter: H100 is a generation-aligned GPU baseline for TPU v5e, A100 is a practical fallback baseline, and P100/V100 should be treated as historical or availability-driven school baselines.

In [ ]:
def run_command(args: List[str], timeout: float = 5.0) -> str:
    """Run a system command and return stdout, or an empty string if unavailable."""
    try:
        completed = subprocess.run(args, capture_output=True, text=True, timeout=timeout, check=False)
        return completed.stdout.strip()
    except Exception:
        return ""


def parse_lscpu_model(lscpu_text: str) -> str:
    for line in lscpu_text.splitlines():
        if line.lower().startswith("model name:"):
            return line.split(":", 1)[1].strip()
    return ""


def detect_cpu_info() -> Dict[str, Any]:
    lscpu_text = run_command(["lscpu"])
    model = parse_lscpu_model(lscpu_text)
    if not model:
        model = run_command(["sysctl", "-n", "machdep.cpu.brand_string"])
    if not model:
        model = platform.processor() or platform.machine() or "Unknown CPU"
    return {
        "model": model,
        "machine": platform.machine(),
        "processor": platform.processor(),
        "lscpu": lscpu_text[:4000],
    }


def detect_gpu_info() -> Dict[str, Any]:
    query = ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
    raw = run_command(query)
    gpus = []
    for line in raw.splitlines():
        parts = [part.strip() for part in line.split(",")]
        if parts and parts[0]:
            gpus.append({"name": parts[0], "memory_total": parts[1] if len(parts) > 1 else "unknown"})
    return {"available": bool(gpus), "count": len(gpus), "gpus": gpus, "nvidia_smi": raw}


def get_gcp_metadata(path: str, timeout: float = 0.35) -> str:
    url = "http://metadata.google.internal/computeMetadata/v1/" + path
    request = urllib.request.Request(url, headers={"Metadata-Flavor": "Google"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.read().decode("utf-8")
    except Exception:
        return ""


def detect_cloud_info() -> Dict[str, str]:
    return {
        "gcp_project_id": get_gcp_metadata("project/project-id"),
        "gcp_zone": get_gcp_metadata("instance/zone"),
        "gcp_machine_type": get_gcp_metadata("instance/machine-type"),
        "hostname": platform.node(),
        "platform": platform.platform(),
    }


def detect_tpu_info() -> Dict[str, Any]:
    tpu_devices = [device for device in jax.devices() if device.platform == "tpu"]
    env_keys = [
        "TPU_NAME",
        "TPU_ACCELERATOR_TYPE",
        "TPU_WORKER_ID",
        "CLOUD_TPU_TASK_ID",
        "COLAB_TPU_ADDR",
        "JAX_PLATFORMS",
    ]
    env = {key: os.environ.get(key, "") for key in env_keys if os.environ.get(key, "")}
    platform_versions = sorted({getattr(device, "platform_version", "") for device in tpu_devices if getattr(device, "platform_version", "")})
    return {
        "available": bool(tpu_devices),
        "count": len(tpu_devices),
        "devices": [str(device) for device in tpu_devices],
        "platform_versions": platform_versions,
        "env": env,
    }


def classify_hardware_generation(device_type: str, hardware_model: str) -> str:
    text = hardware_model.lower()
    if device_type == "GPU" and "h100" in text:
        return "H100: generation-aligned GPU baseline for TPU v5e experiments."
    if device_type == "GPU" and "a100" in text:
        return "A100: practical fallback GPU baseline, not generation-aligned with TPU v5e."
    if device_type == "GPU" and ("v100" in text or "p100" in text):
        return "P100/V100: historical or availability-driven baseline, not generation-aligned."
    if device_type == "TPU" and "v5" in text:
        return "TPU v5 family: target TPU generation for this mini benchmark."
    if device_type == "TPU":
        return "TPU detected: label the exact TPU generation before interpreting results."
    if device_type == "CPU":
        return "CPU baseline: useful for correctness and scale reference, not an accelerator peer."
    return "Hardware generation could not be classified automatically."


def detect_environment() -> Dict[str, Any]:
    devices = jax.devices()
    default_backend = jax.default_backend()
    device_type = default_backend.upper()
    cpu_info = detect_cpu_info()
    gpu_info = detect_gpu_info()
    tpu_info = detect_tpu_info()
    cloud_info = detect_cloud_info()

    if default_backend == "gpu" and gpu_info["gpus"]:
        hardware_model = "; ".join(gpu["name"] + " (" + gpu["memory_total"] + ")" for gpu in gpu_info["gpus"])
    elif default_backend == "tpu":
        env_type = tpu_info["env"].get("TPU_ACCELERATOR_TYPE", "")
        versions = "; ".join(tpu_info["platform_versions"])
        hardware_model = env_type or versions or "TPU detected by JAX"
    else:
        hardware_model = cpu_info["model"]

    accelerator_count = len([device for device in devices if device.platform in ("gpu", "tpu")])
    generation_note = classify_hardware_generation(device_type, hardware_model)

    return {
        "python_version": sys.version.split()[0],
        "jax_version": jax.__version__,
        "default_backend": default_backend,
        "device_type": device_type,
        "hardware_model": hardware_model,
        "accelerator_count": accelerator_count,
        "jax_devices": [str(device) for device in devices],
        "cpu_info": cpu_info,
        "gpu_info": gpu_info,
        "tpu_info": tpu_info,
        "cloud_info": cloud_info,
        "hardware_generation_note": generation_note,
    }


ENV = detect_environment()
print(json.dumps(ENV, indent=2))


## Dataset Choice

Official MLPerf-style LLM benchmarks use much larger corpora, real BPE tokenizers, and much bigger models. That is not appropriate for a small educational experiment with a limited GCP credit budget.

This notebook supports three budget-conscious language-modeling datasets, selected with `DATASET_NAME` above:

* **wikitext2** -- ~2M tokens. The official raw split, downloaded as a zip from the WikiText release. Small and fast; meant as a pipeline debugging sanity check before spending accelerator time, not a serious training run.
* **wikitext103** -- ~1e8 tokens. The standard LM benchmark corpus, downloaded as a zip. At `sequence_length=512` and a 124M-parameter model (the `accelerator_stress` profile), this is sized to fit roughly one epoch within a single-chip TPU v5e run in about an hour -- fixed-shape 512-token packing is TPU-friendly.
* **tinystories** -- ~4.7e8 tokens. Simple, templated vocabulary; even 10-30M-parameter models show clearly decreasing loss quickly, which is useful when you want a short experiment to also confirm "training is actually working," not just measure speed. The full train split is nearly 2GB, so this notebook downloads only the first `MAX_TRAIN_TOKENS`/`MAX_TEST_TOKENS` bytes via an HTTP Range request instead of the whole file.

**Tokenization is byte-level** (`VOCAB_SIZE = 256`): each raw UTF-8 byte of the text is one token. This avoids adding a BPE tokenizer dependency and keeps the same code path working across all three datasets, but it is a real simplification -- byte-level tokens are much less information-dense than subword tokens, so `sequence_length=512` covers far fewer words than 512 real BPE tokens would, and tokens/second is not directly comparable to word-tokenizer or BPE-tokenizer benchmarks. See the Interpretation Guide at the end.

Text is read up to `MAX_TRAIN_TOKENS`/`MAX_TEST_TOKENS` bytes from the start of each split (a bounded, reproducible slice, not a random sample) and packed into fixed-length `sequence_length`-token blocks for causal language modeling: for each block, the model input is `tokens[:-1]` and the target is `tokens[1:]` (next-token prediction).

If a dataset cannot be downloaded, the notebook falls back to small synthetic byte-level text and records that fact in the result notes.

In [ ]:
def download_file(url: str, destination: pathlib.Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Some CDNs (e.g. Cloudflare-fronted hosts) reject the default "Python-urllib/x.y" user
    # agent as bot traffic, so use a browser-like one for all dataset downloads.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    # Stream to disk in chunks rather than buffering the whole response in memory -- TinyStories'
    # train split alone is close to 2GB.
    with urllib.request.urlopen(request, timeout=180) as response, destination.open("wb") as handle:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)


def download_file_capped(url: str, destination: pathlib.Path, max_bytes: int) -> None:
    """Like download_file, but requests only the first max_bytes via an HTTP Range header.

    Some of these text files are far larger than any MAX_TRAIN_TOKENS/MAX_TEST_TOKENS budget
    (TinyStories' train split alone is close to 2GB); downloading and discarding the rest would
    waste time and bandwidth for no benefit, since read_capped_bytes only uses the first
    max_bytes anyway. Falls back to a full download if the server ignores the Range request.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    if max_bytes <= 0:
        download_file(url, destination)
        return
    headers = {"User-Agent": "Mozilla/5.0", "Range": f"bytes=0-{max_bytes - 1}"}
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=180) as response, destination.open("wb") as handle:
        remaining = max_bytes
        while remaining > 0:
            chunk = response.read(min(1024 * 1024, remaining))
            if not chunk:
                break
            handle.write(chunk)
            remaining -= len(chunk)


def read_capped_bytes(path: pathlib.Path, max_bytes: int) -> bytes:
    with path.open("rb") as handle:
        return handle.read(max_bytes) if max_bytes > 0 else handle.read()


# --- WikiText-2 / WikiText-103 --------------------------------------------------------------------

WIKITEXT_SOURCES = {
    "wikitext2": {
        "url": "https://wikitext.smerity.com/wikitext-2-raw-v1.zip",
        "archive_name": "wikitext-2-raw-v1.zip",
        "train_path": "wikitext-2-raw/wiki.train.raw",
        "valid_path": "wikitext-2-raw/wiki.valid.raw",
    },
    "wikitext103": {
        "url": "https://wikitext.smerity.com/wikitext-103-raw-v1.zip",
        "archive_name": "wikitext-103-raw-v1.zip",
        "train_path": "wikitext-103-raw/wiki.train.raw",
        "valid_path": "wikitext-103-raw/wiki.valid.raw",
    },
}


def ensure_wikitext_files(data_root: pathlib.Path, spec: Dict[str, str]) -> Tuple[pathlib.Path, pathlib.Path]:
    if not ALLOW_DATASET_DOWNLOAD:
        raise RuntimeError("Dataset download is disabled by ALLOW_DATASET_DOWNLOAD=False.")
    data_root.mkdir(parents=True, exist_ok=True)
    train_file = data_root / spec["train_path"]
    valid_file = data_root / spec["valid_path"]
    if train_file.exists() and valid_file.exists():
        return train_file, valid_file
    archive_path = data_root / spec["archive_name"]
    if not archive_path.exists():
        print(f"Downloading {spec['archive_name']}...")
        download_file(spec["url"], archive_path)
    print("Extracting archive...")
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(data_root)
    return train_file, valid_file


def load_wikitext_sample(dataset_key: str) -> Tuple[bytes, bytes, Dict[str, str]]:
    spec = WIKITEXT_SOURCES[dataset_key]
    train_file, valid_file = ensure_wikitext_files(DATA_ROOT, spec)
    train_bytes = read_capped_bytes(train_file, MAX_TRAIN_TOKENS)
    test_bytes = read_capped_bytes(valid_file, MAX_TEST_TOKENS)
    info = {
        "dataset": f"{dataset_key} sample",
        "notes": f"Downloaded {spec['archive_name']} and used the first {len(train_bytes)} train / {len(test_bytes)} valid bytes as byte-level tokens.",
    }
    return train_bytes, test_bytes, info


# --- TinyStories -----------------------------------------------------------------------------------

TINYSTORIES_TRAIN_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt"
TINYSTORIES_VALID_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"


def ensure_tinystories_files(data_root: pathlib.Path, max_train_bytes: int, max_test_bytes: int) -> Tuple[pathlib.Path, pathlib.Path]:
    if not ALLOW_DATASET_DOWNLOAD:
        raise RuntimeError("Dataset download is disabled by ALLOW_DATASET_DOWNLOAD=False.")
    data_root.mkdir(parents=True, exist_ok=True)
    train_file = data_root / "TinyStories-train.txt"
    valid_file = data_root / "TinyStories-valid.txt"
    if not train_file.exists():
        print("Downloading TinyStories-train.txt (capped slice)...")
        download_file_capped(TINYSTORIES_TRAIN_URL, train_file, max_train_bytes)
    if not valid_file.exists():
        print("Downloading TinyStories-valid.txt (capped slice)...")
        download_file_capped(TINYSTORIES_VALID_URL, valid_file, max_test_bytes)
    return train_file, valid_file


def load_tinystories_sample() -> Tuple[bytes, bytes, Dict[str, str]]:
    train_file, valid_file = ensure_tinystories_files(DATA_ROOT, MAX_TRAIN_TOKENS, MAX_TEST_TOKENS)
    train_bytes = read_capped_bytes(train_file, MAX_TRAIN_TOKENS)
    test_bytes = read_capped_bytes(valid_file, MAX_TEST_TOKENS)
    info = {
        "dataset": "TinyStories sample",
        "notes": f"Downloaded TinyStories text files and used the first {len(train_bytes)} train / {len(test_bytes)} valid bytes as byte-level tokens.",
    }
    return train_bytes, test_bytes, info


# --- Synthetic fallback ------------------------------------------------------------------------------

def make_synthetic_text_bytes(count: int, seed: int) -> bytes:
    rng = np.random.default_rng(seed)
    vocabulary = np.frombuffer(b" abcdefghijklmnopqrstuvwxyz.,\n", dtype=np.uint8)
    return vocabulary[rng.integers(0, len(vocabulary), size=count)].tobytes()


def make_synthetic_dataset() -> Tuple[bytes, bytes, Dict[str, str]]:
    info = {
        "dataset": f"Synthetic byte-level text sample ({DATASET_NAME})",
        "notes": "Dataset download failed or was disabled; using synthetic repeating-vocabulary text.",
    }
    train_bytes = make_synthetic_text_bytes(max(MAX_TRAIN_TOKENS, SEQUENCE_LENGTH * 64), RANDOM_SEED)
    test_bytes = make_synthetic_text_bytes(max(MAX_TEST_TOKENS, SEQUENCE_LENGTH * 64), RANDOM_SEED + 1)
    return train_bytes, test_bytes, info


def load_dataset() -> Tuple[bytes, bytes, Dict[str, str]]:
    try:
        if DATASET_NAME == "wikitext2":
            return load_wikitext_sample("wikitext2")
        elif DATASET_NAME == "wikitext103":
            return load_wikitext_sample("wikitext103")
        elif DATASET_NAME == "tinystories":
            return load_tinystories_sample()
        else:
            raise ValueError(f"Unsupported DATASET_NAME={DATASET_NAME!r}")
    except Exception as exc:
        if not ALLOW_SYNTHETIC_FALLBACK:
            raise
        print("Dataset load failed; falling back to synthetic data.")
        print(str(exc))
        return make_synthetic_dataset()


def bytes_to_token_array(data: bytes) -> np.ndarray:
    return np.frombuffer(data, dtype=np.uint8).astype(np.int32)


def pack_sequences(tokens: np.ndarray, sequence_length: int) -> Tuple[np.ndarray, np.ndarray]:
    usable_length = ((len(tokens) - 1) // sequence_length) * sequence_length
    if usable_length <= 0:
        raise ValueError(
            f"Not enough tokens ({len(tokens)}) to build a single sequence of length {sequence_length}. "
            "Increase MAX_TRAIN_TOKENS/MAX_TEST_TOKENS or lower sequence_length."
        )
    inputs = tokens[:usable_length].reshape(-1, sequence_length)
    targets = tokens[1:usable_length + 1].reshape(-1, sequence_length)
    return inputs, targets


TRAIN_BYTES, TEST_BYTES, DATASET_INFO = load_dataset()
print(DATASET_INFO)

TRAIN_TOKENS = bytes_to_token_array(TRAIN_BYTES)
TEST_TOKENS = bytes_to_token_array(TEST_BYTES)

# X_TRAIN/Y_TRAIN and X_TEST/Y_TEST hold packed (num_sequences, sequence_length) int32 arrays:
# X is the model input, Y is the next-token target (X shifted left by one position).
X_TRAIN, Y_TRAIN = pack_sequences(TRAIN_TOKENS, SEQUENCE_LENGTH)
X_TEST, Y_TEST = pack_sequences(TEST_TOKENS, SEQUENCE_LENGTH)
print(f"Train sequences: {X_TRAIN.shape}, Test sequences: {X_TEST.shape}")


## Model and Benchmark Utilities

A small GPT-style decoder-only transformer: learned token + position embeddings, `N_LAYER` pre-norm transformer blocks (causal multi-head self-attention + GELU MLP), a final layer norm, and a linear head back to `VOCAB_SIZE` logits. Model size (`N_LAYER`, `N_HEAD`, `N_EMBD`) and `SEQUENCE_LENGTH` come from `BENCHMARK_PROFILE` in the config cell above.

In [ ]:
def he_normal(key: Any, shape: Tuple[int, ...], fan_in: int) -> jnp.ndarray:
    values = jax.random.normal(key, shape, dtype=COMPUTE_DTYPE)
    return values * jnp.asarray(math.sqrt(2.0 / fan_in), dtype=COMPUTE_DTYPE)


def init_layer_norm_params() -> Dict[str, jnp.ndarray]:
    return {
        "scale": jnp.ones((N_EMBD,), dtype=COMPUTE_DTYPE),
        "bias": jnp.zeros((N_EMBD,), dtype=COMPUTE_DTYPE),
    }


def init_transformer_layer_params(key: Any) -> Dict[str, Any]:
    keys = jax.random.split(key, 4)
    return {
        "ln1": init_layer_norm_params(),
        "attn_qkv": {
            "w": he_normal(keys[0], (N_EMBD, 3 * N_EMBD), N_EMBD),
            "b": jnp.zeros((3 * N_EMBD,), dtype=COMPUTE_DTYPE),
        },
        "attn_out": {
            "w": he_normal(keys[1], (N_EMBD, N_EMBD), N_EMBD),
            "b": jnp.zeros((N_EMBD,), dtype=COMPUTE_DTYPE),
        },
        "ln2": init_layer_norm_params(),
        "mlp_fc": {
            "w": he_normal(keys[2], (N_EMBD, 4 * N_EMBD), N_EMBD),
            "b": jnp.zeros((4 * N_EMBD,), dtype=COMPUTE_DTYPE),
        },
        "mlp_proj": {
            "w": he_normal(keys[3], (4 * N_EMBD, N_EMBD), 4 * N_EMBD),
            "b": jnp.zeros((N_EMBD,), dtype=COMPUTE_DTYPE),
        },
    }


def init_model_params(key: Any) -> Dict[str, Any]:
    keys = jax.random.split(key, 3 + N_LAYER)
    return {
        "token_embedding": he_normal(keys[0], (VOCAB_SIZE, N_EMBD), N_EMBD),
        "position_embedding": he_normal(keys[1], (SEQUENCE_LENGTH, N_EMBD), N_EMBD),
        "layers": [init_transformer_layer_params(keys[3 + i]) for i in range(N_LAYER)],
        "final_norm": init_layer_norm_params(),
        "lm_head": he_normal(keys[2], (N_EMBD, VOCAB_SIZE), N_EMBD),
    }


def layer_norm(x: jnp.ndarray, scale: jnp.ndarray, bias: jnp.ndarray, eps: float = 1e-5) -> jnp.ndarray:
    mean = jnp.mean(x, axis=-1, keepdims=True)
    variance = jnp.var(x, axis=-1, keepdims=True)
    normalized = (x - mean) / jnp.sqrt(variance + eps)
    return normalized * scale + bias


def causal_self_attention(x: jnp.ndarray, layer_params: Dict[str, Any]) -> jnp.ndarray:
    batch, seq_len, embd = x.shape
    qkv = x @ layer_params["attn_qkv"]["w"] + layer_params["attn_qkv"]["b"]
    qkv = qkv.reshape(batch, seq_len, 3, N_HEAD, HEAD_DIM)
    q = jnp.transpose(qkv[:, :, 0], (0, 2, 1, 3))
    k = jnp.transpose(qkv[:, :, 1], (0, 2, 1, 3))
    v = jnp.transpose(qkv[:, :, 2], (0, 2, 1, 3))

    scale = 1.0 / math.sqrt(HEAD_DIM)
    attn_scores = jnp.einsum("bhqd,bhkd->bhqk", q, k) * scale
    causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))
    attn_scores = jnp.where(causal_mask, attn_scores, jnp.finfo(attn_scores.dtype).min)
    attn_weights = jax.nn.softmax(attn_scores, axis=-1)
    attn_output = jnp.einsum("bhqk,bhkd->bhqd", attn_weights, v)
    attn_output = jnp.transpose(attn_output, (0, 2, 1, 3)).reshape(batch, seq_len, embd)
    return attn_output @ layer_params["attn_out"]["w"] + layer_params["attn_out"]["b"]


def transformer_block(x: jnp.ndarray, layer_params: Dict[str, Any]) -> jnp.ndarray:
    attn_input = layer_norm(x, layer_params["ln1"]["scale"], layer_params["ln1"]["bias"])
    x = x + causal_self_attention(attn_input, layer_params)
    mlp_input = layer_norm(x, layer_params["ln2"]["scale"], layer_params["ln2"]["bias"])
    hidden = jax.nn.gelu(mlp_input @ layer_params["mlp_fc"]["w"] + layer_params["mlp_fc"]["b"])
    mlp_output = hidden @ layer_params["mlp_proj"]["w"] + layer_params["mlp_proj"]["b"]
    return x + mlp_output


def forward(params: Dict[str, Any], tokens: jnp.ndarray) -> jnp.ndarray:
    batch, seq_len = tokens.shape
    token_embeddings = jnp.take(params["token_embedding"], tokens, axis=0)
    position_embeddings = params["position_embedding"][:seq_len]
    x = (token_embeddings + position_embeddings).astype(COMPUTE_DTYPE)
    for layer_params in params["layers"]:
        x = transformer_block(x, layer_params)
    x = layer_norm(x, params["final_norm"]["scale"], params["final_norm"]["bias"])
    logits = x @ params["lm_head"]
    return logits.astype(jnp.float32)


def cross_entropy_loss(logits: jnp.ndarray, targets: jnp.ndarray) -> jnp.ndarray:
    one_hot = jax.nn.one_hot(targets, VOCAB_SIZE)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(jnp.sum(one_hot * log_probs, axis=-1))


@jax.jit
def train_step(params: Dict[str, Any], x: jnp.ndarray, y: jnp.ndarray, learning_rate: jnp.ndarray):
    def loss_fn(current_params):
        logits = forward(current_params, x)
        return cross_entropy_loss(logits, y)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updated = jax.tree_util.tree_map(lambda p, g: p - learning_rate * g, params, grads)
    return updated, loss


@jax.jit
def inference_step(params: Dict[str, Any], x: jnp.ndarray) -> jnp.ndarray:
    # Full-sequence forward pass (like a training/eval forward pass over a batch of sequences),
    # not autoregressive token-by-token generation. See the Interpretation Guide at the end.
    return forward(params, x)


def block_until_ready_tree(value: Any) -> Any:
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def make_device_batches(
    sequences_x: np.ndarray,
    sequences_y: np.ndarray,
    batch_size: int,
    num_batches: int,
    seed: int,
) -> List[Tuple[jnp.ndarray, jnp.ndarray]]:
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(sequences_x), size=(num_batches, batch_size))
    batches = []
    for batch_indices in indices:
        x_batch = jax.device_put(sequences_x[batch_indices])
        y_batch = jax.device_put(sequences_y[batch_indices])
        batches.append((x_batch, y_batch))
    return batches


def compile_jitted(jitted_function: Any, *args: Any) -> Tuple[Any, float, str]:
    start = time.perf_counter()
    try:
        compiled = jitted_function.lower(*args).compile()
        compile_time = time.perf_counter() - start
        return compiled, compile_time, "explicit_lower_compile"
    except AttributeError:
        # Older JAX versions may not expose lower(). In that case, the first call includes compilation.
        return jitted_function, math.nan, "first_call_includes_compile"


def short_error(exc: Exception, max_chars: int = 350) -> str:
    text = " ".join(str(exc).split())
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text


def is_probable_oom(exc: Exception) -> bool:
    text = str(exc).lower()
    markers = ["out of memory", "oom", "resource exhausted", "allocation", "memory"]
    return any(marker in text for marker in markers)


def get_gpu_memory_used_mb() -> Optional[float]:
    raw = run_command(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], timeout=2.0)
    values = []
    for line in raw.splitlines():
        try:
            values.append(float(line.strip()))
        except ValueError:
            pass
    return max(values) if values else None


def get_process_memory_mb() -> Optional[float]:
    try:
        import psutil

        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    except Exception:
        try:
            import resource

            usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            if sys.platform == "darwin":
                return usage / (1024 ** 2)
            return usage / 1024
        except Exception:
            return None


def get_memory_usage_mb() -> Optional[float]:
    if ENV["device_type"] == "GPU":
        return get_gpu_memory_used_mb()
    if ENV["device_type"] == "TPU":
        return None
    return get_process_memory_mb()


def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def base_result_row(task_type: str, batch_size: int) -> Dict[str, Any]:
    return {
        "timestamp": now_iso(),
        "run_id": RUN_ID,
        "device_type": ENV["device_type"],
        "hardware_model": ENV["hardware_model"],
        "accelerator_count": ENV["accelerator_count"],
        "framework": FRAMEWORK_NAME,
        "framework_version": jax.__version__,
        "task_type": task_type,
        "dataset": DATASET_INFO["dataset"],
        "model": MODEL_NAME,
        "batch_size": batch_size,
        "sequence_length": SEQUENCE_LENGTH,
        "vocab_size": VOCAB_SIZE,
        "compile_time_seconds": math.nan,
        "warm_up_time_seconds": math.nan,
        "average_latency_ms": math.nan,
        "median_latency_ms": math.nan,
        "throughput_samples_per_second": math.nan,
        "tokens_per_second": math.nan,
        "total_wall_time_seconds": math.nan,
        "number_of_iterations": 0,
        "first_run_overhead_seconds": math.nan,
        "memory_used_mb": math.nan,
        "numeric_precision": str(COMPUTE_DTYPE),
        "hardware_generation_note": ENV["hardware_generation_note"],
        "status": "pending",
        "notes": DATASET_INFO["notes"],
    }


def append_results_to_csv(rows: List[Dict[str, Any]], path: pathlib.Path) -> None:
    if not rows:
        return
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open("a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=RESULT_COLUMNS, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def clear_jax_caches_if_available() -> None:
    try:
        jax.clear_caches()
    except Exception:
        pass


print("Model parameter count:")
_params_preview = init_model_params(jax.random.PRNGKey(RANDOM_SEED))
_param_count = sum(int(np.prod(leaf.shape)) for leaf in jax.tree_util.tree_leaves(_params_preview))
print(_param_count)


## Part 1: Training Benchmark

Warning: This cell may use billable accelerator resources. Confirm that you are running on the intended runtime and that your GCP quota and budget are appropriate before continuing.

The training benchmark compiles the JAX training step for each batch size (number of sequences per batch), runs separate warm-up steps, then measures a small number of SGD steps of causal-LM next-token-prediction training. Failed batch sizes are recorded instead of crashing the notebook.

In [ ]:
def run_training_for_batch_size(batch_size: int) -> Tuple[Dict[str, Any], Optional[Dict[str, Any]]]:
    row = base_result_row("training", batch_size)
    params = None
    compile_time = math.nan
    first_run_time = math.nan
    try:
        total_iterations = TRAIN_WARMUP_STEPS + TRAIN_MEASURE_STEPS
        batches = make_device_batches(X_TRAIN, Y_TRAIN, batch_size, total_iterations, RANDOM_SEED + batch_size)
        params = jax.device_put(init_model_params(jax.random.PRNGKey(RANDOM_SEED)))
        learning_rate = jnp.asarray(LEARNING_RATE, dtype=jnp.float32)
        memory_before = get_memory_usage_mb()

        compiled_train_step, compile_time, compile_mode = compile_jitted(
            train_step,
            params,
            batches[0][0],
            batches[0][1],
            learning_rate,
        )

        first_start = time.perf_counter()
        params, loss = compiled_train_step(params, batches[0][0], batches[0][1], learning_rate)
        block_until_ready_tree((params, loss))
        first_run_time = time.perf_counter() - first_start

        warmup_start = time.perf_counter()
        for batch_index in range(1, TRAIN_WARMUP_STEPS):
            params, loss = compiled_train_step(params, batches[batch_index][0], batches[batch_index][1], learning_rate)
            block_until_ready_tree((params, loss))
        warmup_time = first_run_time + (time.perf_counter() - warmup_start)

        step_times = []
        measured_start = time.perf_counter()
        for batch_index in range(TRAIN_WARMUP_STEPS, total_iterations):
            step_start = time.perf_counter()
            params, loss = compiled_train_step(params, batches[batch_index][0], batches[batch_index][1], learning_rate)
            block_until_ready_tree((params, loss))
            step_times.append(time.perf_counter() - step_start)
        measured_wall = time.perf_counter() - measured_start

        average_step = statistics.mean(step_times)
        median_step = statistics.median(step_times)
        memory_after = get_memory_usage_mb()
        total_wall = (0.0 if math.isnan(compile_time) else compile_time) + warmup_time + measured_wall
        first_run_overhead = first_run_time if math.isnan(compile_time) else compile_time + first_run_time
        sequences_per_second = batch_size / average_step

        row.update(
            {
                "compile_time_seconds": compile_time,
                "warm_up_time_seconds": warmup_time,
                "average_latency_ms": average_step * 1000.0,
                "median_latency_ms": median_step * 1000.0,
                "throughput_samples_per_second": sequences_per_second,
                "tokens_per_second": sequences_per_second * SEQUENCE_LENGTH,
                "total_wall_time_seconds": total_wall,
                "number_of_iterations": TRAIN_MEASURE_STEPS,
                "first_run_overhead_seconds": first_run_overhead,
                "memory_used_mb": memory_after if memory_after is not None else math.nan,
                "status": "success",
                "notes": f"{DATASET_INFO['notes']} compile_mode={compile_mode}; memory_before_mb={memory_before}; memory_after_mb={memory_after}; final_loss={float(loss):.5f}",
            }
        )
        return row, params
    except Exception as exc:
        row.update(
            {
                "status": "failed_oom" if is_probable_oom(exc) else "failed",
                "notes": f"{DATASET_INFO['notes']} error={short_error(exc)}",
                "memory_used_mb": get_memory_usage_mb() or math.nan,
            }
        )
        gc.collect()
        clear_jax_caches_if_available()
        return row, None


def run_training_benchmark() -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:
    results = []
    last_successful_params = None
    for batch_size in BATCH_SIZES:
        print(f"Training batch size {batch_size} (sequences), {batch_size * SEQUENCE_LENGTH} tokens/batch...")
        row, params = run_training_for_batch_size(batch_size)
        results.append(row)
        if row["status"] == "success" and params is not None:
            last_successful_params = params
        print({key: row[key] for key in ["batch_size", "status", "average_latency_ms", "tokens_per_second", "compile_time_seconds"]})
    return results, last_successful_params


if RUN_TRAINING_BENCHMARK:
    TRAINING_RESULTS, TRAINED_PARAMS = run_training_benchmark()
    append_results_to_csv(TRAINING_RESULTS, TRAINING_RESULTS_PATH)
    append_results_to_csv(TRAINING_RESULTS, ALL_RESULTS_PATH)
    print(f"Saved training results to {TRAINING_RESULTS_PATH} and {ALL_RESULTS_PATH}")
else:
    TRAINING_RESULTS = []
    TRAINED_PARAMS = None
    print("Training benchmark skipped by RUN_TRAINING_BENCHMARK=False")


## Part 2: Inference Benchmark

Warning: This cell may use billable accelerator resources. Confirm that you are running on the intended runtime and that your GCP quota and budget are appropriate before continuing.

The inference benchmark is separated from training. It measures first-run overhead, warm-up time, steady-state per-batch latency, and throughput for each batch size.

Important: this measures a **full-sequence forward pass** over a batch of `sequence_length`-token sequences (the same shape of computation as one training forward pass, without backprop) -- not autoregressive, KV-cache-based, token-by-token text generation. Real-world LLM serving latency for generation is a very different workload; do not read these numbers as generation latency.

In [ ]:
def get_params_for_inference() -> Dict[str, Any]:
    if "TRAINED_PARAMS" in globals() and TRAINED_PARAMS is not None:
        return TRAINED_PARAMS
    return jax.device_put(init_model_params(jax.random.PRNGKey(RANDOM_SEED)))


def run_inference_for_batch_size(params: Dict[str, Any], batch_size: int) -> Dict[str, Any]:
    row = base_result_row("inference", batch_size)
    compile_time = math.nan
    first_run_time = math.nan
    try:
        total_iterations = INFERENCE_WARMUP_STEPS + INFERENCE_MEASURE_STEPS
        batches = make_device_batches(X_TEST, Y_TEST, batch_size, total_iterations, RANDOM_SEED + 1000 + batch_size)
        memory_before = get_memory_usage_mb()

        compiled_inference_step, compile_time, compile_mode = compile_jitted(inference_step, params, batches[0][0])

        first_start = time.perf_counter()
        logits = compiled_inference_step(params, batches[0][0])
        block_until_ready_tree(logits)
        first_run_time = time.perf_counter() - first_start

        warmup_start = time.perf_counter()
        for batch_index in range(1, INFERENCE_WARMUP_STEPS):
            logits = compiled_inference_step(params, batches[batch_index][0])
            block_until_ready_tree(logits)
        warmup_time = first_run_time + (time.perf_counter() - warmup_start)

        step_times = []
        measured_start = time.perf_counter()
        for batch_index in range(INFERENCE_WARMUP_STEPS, total_iterations):
            step_start = time.perf_counter()
            logits = compiled_inference_step(params, batches[batch_index][0])
            block_until_ready_tree(logits)
            step_times.append(time.perf_counter() - step_start)
        measured_wall = time.perf_counter() - measured_start

        average_step = statistics.mean(step_times)
        median_step = statistics.median(step_times)
        memory_after = get_memory_usage_mb()
        total_wall = (0.0 if math.isnan(compile_time) else compile_time) + warmup_time + measured_wall
        first_run_overhead = first_run_time if math.isnan(compile_time) else compile_time + first_run_time
        sequences_per_second = batch_size / average_step

        row.update(
            {
                "compile_time_seconds": compile_time,
                "warm_up_time_seconds": warmup_time,
                "average_latency_ms": average_step * 1000.0,
                "median_latency_ms": median_step * 1000.0,
                "throughput_samples_per_second": sequences_per_second,
                "tokens_per_second": sequences_per_second * SEQUENCE_LENGTH,
                "total_wall_time_seconds": total_wall,
                "number_of_iterations": INFERENCE_MEASURE_STEPS,
                "first_run_overhead_seconds": first_run_overhead,
                "memory_used_mb": memory_after if memory_after is not None else math.nan,
                "status": "success",
                "notes": f"{DATASET_INFO['notes']} compile_mode={compile_mode}; memory_before_mb={memory_before}; memory_after_mb={memory_after}",
            }
        )
        return row
    except Exception as exc:
        row.update(
            {
                "status": "failed_oom" if is_probable_oom(exc) else "failed",
                "notes": f"{DATASET_INFO['notes']} error={short_error(exc)}",
                "memory_used_mb": get_memory_usage_mb() or math.nan,
            }
        )
        gc.collect()
        clear_jax_caches_if_available()
        return row


def run_inference_benchmark() -> List[Dict[str, Any]]:
    params = get_params_for_inference()
    results = []
    for batch_size in BATCH_SIZES:
        print(f"Inference batch size {batch_size} (sequences), {batch_size * SEQUENCE_LENGTH} tokens/batch...")
        row = run_inference_for_batch_size(params, batch_size)
        results.append(row)
        print({key: row[key] for key in ["batch_size", "status", "average_latency_ms", "tokens_per_second", "compile_time_seconds"]})
    return results


if RUN_INFERENCE_BENCHMARK:
    INFERENCE_RESULTS = run_inference_benchmark()
    append_results_to_csv(INFERENCE_RESULTS, INFERENCE_RESULTS_PATH)
    append_results_to_csv(INFERENCE_RESULTS, ALL_RESULTS_PATH)
    print(f"Saved inference results to {INFERENCE_RESULTS_PATH} and {ALL_RESULTS_PATH}")
else:
    INFERENCE_RESULTS = []
    print("Inference benchmark skipped by RUN_INFERENCE_BENCHMARK=False")


## Visualizations

Charts are generated from `lm_benchmark_results_all.csv`. If you run this notebook on multiple runtimes and datasets, combine or copy the CSV rows into that file before re-running this section -- the `dataset` column separates series so you can compare across both device and dataset.

Both a sequences/second view (`throughput_samples_per_second`) and a tokens/second view (`tokens_per_second` = sequences/second x `sequence_length`) are plotted, since tokens/second is the more standard way to compare LM throughput across profiles with different `sequence_length`.

In [ ]:
def load_csv_rows(path: pathlib.Path) -> List[Dict[str, str]]:
    if not path.exists():
        return []
    with path.open("r", newline="") as handle:
        return list(csv.DictReader(handle))


def to_float(value: Any) -> float:
    try:
        result = float(value)
        return result
    except Exception:
        return math.nan


def to_int(value: Any) -> int:
    try:
        return int(float(value))
    except Exception:
        return 0


def compact_hardware_name(name: str, max_len: int = 46) -> str:
    text = name.replace("NVIDIA ", "").replace("Google ", "")
    return text if len(text) <= max_len else text[:max_len - 3] + "..."


def device_label(row: Dict[str, Any]) -> str:
    return f"{row.get('device_type', 'unknown')} | {compact_hardware_name(row.get('hardware_model', 'unknown'))} | {row.get('dataset', 'unknown')}"


def successful_rows(rows: Iterable[Dict[str, Any]], task_type: Optional[str] = None) -> List[Dict[str, Any]]:
    selected = []
    for row in rows:
        if row.get("status") != "success":
            continue
        if task_type is not None and row.get("task_type") != task_type:
            continue
        selected.append(row)
    return selected


def plot_metric_by_batch(rows: List[Dict[str, Any]], task_type: str, metric: str, ylabel: str, title: str) -> None:
    task_rows = successful_rows(rows, task_type)
    if not task_rows:
        print(f"No successful {task_type} rows available for {title}.")
        return

    grouped: Dict[str, List[Tuple[int, float]]] = {}
    for row in task_rows:
        x = to_int(row.get("batch_size"))
        y = to_float(row.get(metric))
        if x > 0 and not math.isnan(y):
            grouped.setdefault(device_label(row), []).append((x, y))

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    for label, points in sorted(grouped.items()):
        points = sorted(points)
        ax.plot([point[0] for point in points], [point[1] for point in points], marker="o", linewidth=2, label=label)

    ax.set_title(title)
    ax.set_xlabel("Batch size (sequences)")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted({to_int(row.get("batch_size")) for row in task_rows if to_int(row.get("batch_size")) > 0}))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()


def plot_compile_overhead(rows: List[Dict[str, Any]]) -> None:
    task_rows = successful_rows(rows)
    if not task_rows:
        print("No successful rows available for compile overhead chart.")
        return

    grouped: Dict[str, List[Tuple[int, float]]] = {}
    for row in task_rows:
        x = to_int(row.get("batch_size"))
        y = to_float(row.get("compile_time_seconds"))
        if x > 0 and not math.isnan(y):
            label = f"{row.get('task_type', 'task')} | {device_label(row)}"
            grouped.setdefault(label, []).append((x, y))

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    for label, points in sorted(grouped.items()):
        points = sorted(points)
        ax.plot([point[0] for point in points], [point[1] for point in points], marker="o", linewidth=2, label=label)

    ax.set_title("JAX/XLA Compile Time by Batch Size")
    ax.set_xlabel("Batch size (sequences)")
    ax.set_ylabel("Compile time (seconds)")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted({to_int(row.get("batch_size")) for row in task_rows if to_int(row.get("batch_size")) > 0}))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()


ALL_ROWS = load_csv_rows(ALL_RESULTS_PATH)
print(f"Loaded {len(ALL_ROWS)} rows from {ALL_RESULTS_PATH}")

plot_metric_by_batch(
    ALL_ROWS,
    "training",
    "tokens_per_second",
    "Tokens per second",
    "Training Token Throughput by Batch Size, Device, and Dataset",
)
plot_metric_by_batch(
    ALL_ROWS,
    "training",
    "average_latency_ms",
    "Average step time (ms per batch)",
    "Training Step Time by Batch Size, Device, and Dataset",
)
plot_metric_by_batch(
    ALL_ROWS,
    "inference",
    "tokens_per_second",
    "Tokens per second",
    "Inference Token Throughput by Batch Size, Device, and Dataset",
)
plot_metric_by_batch(
    ALL_ROWS,
    "inference",
    "median_latency_ms",
    "Median latency (ms per batch)",
    "Inference Latency by Batch Size, Device, and Dataset",
)
plot_compile_overhead(ALL_ROWS)


## Optional Cost-Efficiency Chart

If you know your hourly prices, fill in `HOURLY_COST_USD_BY_DEVICE_LABEL` below. Prices vary by region, reservation, spot/preemptible settings, and date, so this notebook does not hard-code prices.

Use labels printed by `device_label(row)` in the previous section, or inspect `lm_benchmark_results_all.csv`.

In [ ]:
HOURLY_COST_USD_BY_DEVICE_LABEL = {
    # Example:
    # "GPU | H100 80GB HBM3 | wikitext103 sample": 8.00,
    # "TPU | TPU v5e | wikitext103 sample": 1.20,
    # "CPU | Intel(R) Xeon(R) ... | wikitext103 sample": 0.30,
}


def plot_cost_efficiency(rows: List[Dict[str, Any]], hourly_costs: Dict[str, float], task_type: str) -> None:
    task_rows = successful_rows(rows, task_type)
    points = []
    for row in task_rows:
        label = device_label(row)
        hourly_cost = hourly_costs.get(label)
        tokens_per_second = to_float(row.get("tokens_per_second"))
        if hourly_cost and tokens_per_second > 0:
            tokens_per_dollar = tokens_per_second * 3600.0 / hourly_cost
            points.append((label, to_int(row.get("batch_size")), tokens_per_dollar))

    if not points:
        print(f"Cost-efficiency chart skipped for {task_type}; provide hourly costs to enable it.")
        return

    labels = [f"{label}\nbs={batch_size}" for label, batch_size, _ in points]
    values = [value for _, _, value in points]
    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.bar(range(len(values)), values)
    ax.set_title(f"{task_type.title()} Cost Efficiency")
    ax.set_ylabel("Tokens per dollar")
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


plot_cost_efficiency(ALL_ROWS, HOURLY_COST_USD_BY_DEVICE_LABEL, "training")
plot_cost_efficiency(ALL_ROWS, HOURLY_COST_USD_BY_DEVICE_LABEL, "inference")


## Summary Table

The summary table reports the best token-throughput row for each task, device, and dataset combination. It is most useful after you have combined CPU, GPU, and TPU CSV rows across all three datasets.

In [ ]:
def markdown_table(headers: List[str], rows: List[List[str]]) -> str:
    header_line = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header_line, separator] + body)


def build_summary_table(rows: List[Dict[str, Any]]) -> str:
    grouped: Dict[Tuple[str, str], List[Dict[str, Any]]] = {}
    for row in successful_rows(rows):
        key = (row.get("task_type", "unknown"), device_label(row))
        grouped.setdefault(key, []).append(row)

    summary_rows = []
    for (task_type, label), group in sorted(grouped.items()):
        best = max(group, key=lambda row: to_float(row.get("tokens_per_second")))
        latency_values = [to_float(row.get("median_latency_ms")) for row in group if not math.isnan(to_float(row.get("median_latency_ms")))]
        best_latency = min(latency_values) if latency_values else math.nan
        summary_rows.append(
            [
                task_type,
                label,
                str(best.get("batch_size")),
                f"{to_float(best.get('tokens_per_second')):.2f}",
                f"{best_latency:.3f}",
                f"{to_float(best.get('compile_time_seconds')):.3f}",
                best.get("hardware_generation_note", ""),
            ]
        )

    if not summary_rows:
        return "No successful benchmark rows are available yet."

    return markdown_table(
        [
            "Task",
            "Device | Dataset",
            "Best batch size",
            "Best tokens/sec",
            "Best median latency ms",
            "Compile seconds at best throughput",
            "Generation note",
        ],
        summary_rows,
    )


summary = build_summary_table(ALL_ROWS)
try:
    from IPython.display import Markdown, display

    display(Markdown(summary))
except Exception:
    print(summary)


## Interpretation Guide

Use these results carefully:

1. TPU/GPU comparison depends heavily on model size and sequence length, not just dataset. `accelerator_stress`'s 124M-parameter, sequence_length=512 model is closer to a real (if small) LLM pretraining shape than the `quick`/`expanded` profiles, and is far more likely to show a clean accelerator advantage.
2. Byte-level tokenization is a real simplification, not a neutral one. Because each token is one raw byte, `tokens_per_second` here is not directly comparable to `tokens_per_second` reported by BPE-tokenized benchmarks (real BPE tokens each encode roughly 3-4x more text). Treat this notebook's numbers as internally consistent (comparable across CPU/GPU/TPU and across WikiText-2/103/TinyStories) but not as directly comparable to external published LM throughput numbers.
3. Dataset choice changes what a good result looks like. WikiText-2 is a pipeline sanity check, not a real training run -- do not draw hardware conclusions from it. WikiText-103 is the closest of the three to a standard LM benchmark scale. TinyStories has a much simpler distribution, so even small models will show visibly decreasing training loss (`final_loss` in the notes column) over the measured steps -- useful for confirming the training loop is actually learning, not just running.
4. Training and inference results should not be mixed. Training includes backpropagation and optimizer updates; inference here is a full-sequence forward pass only.
5. The inference benchmark is NOT autoregressive generation latency. It measures one forward pass over a full `sequence_length`-token batch (as in an evaluation/scoring pass), not token-by-token decoding with a KV cache. Real-world LLM serving latency is dominated by the autoregressive decode loop, which this notebook does not implement.
6. Batch size changes the story. Larger batches can improve throughput by filling the accelerator, but may increase per-batch latency and memory usage -- and memory usage scales with `batch_size x sequence_length x n_embd`, so OOM failures (recorded, not crashes) are more likely here than in the vision notebook at the same batch sizes.
7. JAX/XLA compile time matters, and matters more here than for the small CNN. Larger transformer graphs can take noticeably longer to compile; for short benchmark runs this overhead can dominate total wall time.
8. CPU is a baseline for correctness and scale intuition. Do not expect CPU to complete `accelerator_stress`-sized runs quickly, or possibly at all within a reasonable time -- use `quick` on CPU.
9. Hardware generation must be labeled. H100 is a generation-aligned GPU baseline for TPU v5e. A100 is useful when H100 is unavailable, but should be labeled as a practical fallback. P100 and V100 should be treated as historical or availability-driven baselines, not generation-aligned baselines.
10. Precision matters. This notebook defaults to float32 for simplicity. Real LLM pretraining almost always uses bfloat16 or mixed precision; a float32-only comparison likely understates TPU/GPU throughput relative to typical production configurations.
11. Small mini benchmarks should not be overgeneralized to full-scale MLPerf or production LLM pretraining. They are useful for teaching concepts, identifying bottlenecks, and practicing measurement discipline.

## Suggested Next Steps

1. Run CPU first with `wikitext2` + `quick` to validate the pipeline, then move to accelerators.
2. Run TPU v5e with `wikitext103` + `accelerator_stress` and the same batch sizes/iteration counts as your GPU run.
3. Run H100 only after the workflow is validated.
4. Try `tinystories` with a smaller model (`expanded` profile) to see loss actually drop within the measured steps, then compare its tokens/second against `wikitext103` at the same profile.
5. Combine the CSV files across datasets and re-run the visualization cells; the `dataset` column keeps series separated.
6. Add current hourly prices to the optional cost-efficiency section.
7. Try an explicitly labeled bfloat16 experiment and compare compile time, throughput, and loss behavior.
8. For a more realistic (if more complex) experiment, replace the byte-level tokenizer with a real BPE tokenizer (e.g. `tiktoken`), keeping the same packing/benchmark harness.
9. For image-classification workloads instead of language modeling, see the companion `mlperf_style_vision_benchmark.ipynb` notebook (Fashion-MNIST, CIFAR-10, Imagenette).